# Embeddings demo

## 1. Calculate and store embeddings of overview field

Clone repo

In [5]:
!git clone https://github.com/tkubica12/ai-demos.git

Cloning into 'ai-demos'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 123 (delta 0), reused 3 (delta 0), pack-reused 120
Receiving objects: 100% (123/123), 60.13 MiB | 22.57 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Updating files: 100% (78/78), done.


Install dependencies

In [1]:
!pip install python-dotenv openai

     |████████████████████████████████| 55 kB 3.6 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
    Preparing wheel metadata ... done
  Created wheel for openai: filename=openai-0.26.4-py3-none-any.whl size=67722 sha256=11cdc7e3d170c01ef7a239371fce9c94849288b1cf93d23a56372f1d79f908f5
  Stored in directory: /home/azureuser/.cache/pip/wheels/2b/d8/4e/268f029bd3277c1dd9e8781a0e0296e0a63822665bfa2429fc
Successfully built openai


Load Azure OpenAI settings and keys

In [1]:
# Store Azure OpenAI key in OPENAI_API_KEY and url in OPENAI_API_URL environmental variable

# Assign envs directly in notebook
# %env OPENAI_API_KEY="mykey"
# %env OPENAI_API_URL="myendpoint"

# Or load from .env file you store on Jupyter server
%load_ext dotenv
%dotenv

Load data into Pandas DataFrame

In [2]:
import pandas as pd
import json

In [3]:
with open("./ai-demos/openai/datasets/movies.json", 'r') as f:
        lines = [json.loads(line) for line in f]

df = pd.DataFrame(lines)

In [4]:
# Print overview of Pulp Fiction
df[(df['title'] == 'Pulp Fiction: Historky z podsvětí')].overview.values[0]

'Pulp Fiction je brutální, ale brilantní černá komedie, ve které se prolínají tři příběhy ze života bezvýznamných zločinců, v podstatě tvořící jeden příběh, do kterého nonkonformní režisér Tarantino zapojil do té doby nevídanou kreativní představivost. Děj Pulp Fiction tvoří několik příběhů z losangeleského gangsterského prostředí, které se v několika scénách spojují a prolínají. Jejich společným prvkem jsou postavy Marsella Wallace (Ving Rhames), Vincenta Vegy (John Travolta) a Miy Wallacové (Uma Thurman), což jsou jediné tři postavy, které se vyskytují ve všech povídkách. Příběhy jsou plné bizarních dějových zvratů, narážek na klasické béčkové filmy i postmoderních prvků. Podsvětí s násilím postrádajícím veškerou logiku je ve filmu nejen teatrální, ale i humorné. Nejkultovnější záležitost 90. let získala 7 oscarových nominací, Zlatý Glóbus a mnoho dalších prestižních ocenění.'

Try scoring on single record

In [ ]:
def get_embedding(text, model="text-embedding-ada-002"):
   text = text.replace("\n", " ")
   return openai.Embedding.create(input = [text], model=model)['data'][0]['embedding']
 
df['ada_embedding'] = df.combined.apply(lambda x: get_embedding(x, model='text-embedding-ada-002'))
df.to_csv('output/embedded_1k_reviews.csv', index=False)

In [6]:
import openai

openai.api_type = "azure"
openai.api_base = os.getenv("OPENAI_API_URL")
openai.api_version = "2022-12-01"
openai.api_key = os.getenv("OPENAI_API_KEY")

In [10]:
deployment = "text-similarity-ada-001"
text = df[(df['title'] == 'Pulp Fiction: Historky z podsvětí')].overview.values[0]

openai.Embedding.create(input = [text], deployment_id=deployment)['data'][0]['embedding']

[0.013160265982151031,
 0.0013550352305173874,
 0.009238876402378082,
 0.05584387481212616,
 0.008776932023465633,
 0.01851881481707096,
 0.02350780740380287,
 0.004645101726055145,
 0.05165558308362961,
 -0.0012376244412735105,
 -0.03447127342224121,
 -0.013026815839111805,
 -0.023713115602731705,
 -0.06479531526565552,
 -0.014156011864542961,
 -0.03434808924794197,
 -0.022604450583457947,
 0.025827791541814804,
 0.01708165556192398,
 0.002507328288629651,
 0.02617681585252285,
 -0.0048837726935744286,
 0.01552131213247776,
 -0.008468969725072384,
 0.054694145917892456,
 0.02712123468518257,
 0.033752694725990295,
 0.02562248334288597,
 -0.017399882897734642,
 -0.0028537861071527004,
 -0.051696643233299255,
 0.0020710481330752373,
 0.013786456547677517,
 0.061058707535266876,
 -0.0173177607357502,
 0.0009809889597818255,
 -0.03744824603199959,
 0.026525840163230896,
 0.001448707072995603,
 -0.01807740144431591,
 -0.022419672459363937,
 0.005902615375816822,
 0.014227868989109993,
 0.0

Generate and store embeddings

In [17]:
deployment = "text-similarity-ada-001"

# Define lambda function
def get_embedding(text, deployment):
   text = text.replace("\n", " ")
   return openai.Embedding.create(input = [text], deployment_id=deployment)['data'][0]['embedding']
 
df['embeddings'] = df['overview'].apply(lambda x: get_embedding(text=x, deployment=deployment))

/tmp/ipykernel_20428/634747246.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edf['embedding'] = edf['overview'].apply(lambda x: get_embedding(text=x, deployment=deployment))


Store output

In [5]:
df.to_json(f'./ai-demos/openai/datasets/movies_{deployment}.json', orient='records')